In [1]:
import pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

import scanpy as sc
import sklearn
import numpy as np
import pandas as pd
import statsmodels as stat
import matplotlib as plt
import seaborn as sns
import anndata as ad

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [6]:
raw_counts_df = pd.read_csv("GSE147523_raw_counts_GRCh38.p13_NCBI.tsv.gz", sep="\t", index_col=0)
annotation_df = pd.read_csv("GSE147523_HumanAnnotate.tsv.gz", sep="\t")

/var/folders/32/q9wbsvsd5bjdyq65l_r635n80000gn/T/ipykernel_52541/3707182350.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  annotation_df = pd.read_csv("GSE147523_HumanAnnotate.tsv.gz", sep="\t")


In [3]:
# Metadata mapping for GSE147523

metadata_dict = {
    # 3 Normal Pediatric Controls
    "GSM4432535": "Control",
    "GSM4432536": "Control",
    "GSM4432537": "Control",
    
    # 19 JMML Samples (mapping all other GSMs)
    "GSM4432517": "JMML", "GSM4432518": "JMML", "GSM4432519": "JMML",
    "GSM4432520": "JMML", "GSM4432521": "JMML", "GSM4432522": "JMML",
    "GSM4432523": "JMML", "GSM4432524": "JMML", "GSM4432525": "JMML",
    "GSM4432526": "JMML", "GSM4432527": "JMML", "GSM4432528": "JMML",
    "GSM4432529": "JMML", "GSM4432530": "JMML", "GSM4432531": "JMML",
    "GSM4432532": "JMML", "GSM4432533": "JMML", "GSM4432534": "JMML",
}

metadata = pd.DataFrame.from_dict(metadata_dict, orient='index', columns=['condition'])
metadata.index.name = 'sample_id'
metadata['condition'] = metadata['condition'].astype('category')

print(metadata['condition'].value_counts())

condition
JMML       18
Control     3
Name: count, dtype: int64


In [7]:
# 1. Setting Indices
if 'GeneID' in raw_counts_df.columns:
    counts_data = raw_counts_df.set_index('GeneID')
else:
    counts_data = raw_counts_df.copy()

print('Step 1 done')

# 2. Align and Sterilize counts
aligned_samples = [col for col in counts_data.columns if col in metadata.index]
counts_data = counts_data[aligned_samples]

clean_genes = [str(g).strip() for g in counts_data.index]
clean_samples = [str(s).strip() for s in counts_data.columns]
pure_array = counts_data.to_numpy(dtype=np.int64)

counts_sterile = pd.DataFrame(
    pure_array,
    index=clean_genes,
    columns=clean_samples
)

metadata_sterile = metadata.loc[counts_sterile.columns].copy()

print('Step 2 done')

# 3. Run standard pyDESeq2
dds = DeseqDataSet(
    counts=counts_sterile.T,          # Samples must be rows
    metadata=metadata_sterile, 
    design_factors="condition",
    refit_cooks=True
)

dds.deseq2()   # This native command will now run perfectly!

print('Step 3 done')

Step 1 done
Step 2 done
Using None as control genes, passed at DeseqDataSet initialization


/var/folders/32/q9wbsvsd5bjdyq65l_r635n80000gn/T/ipykernel_52541/1655038566.py:28: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(
Fitting size factors...
... done in 0.03 seconds.

Fitting dispersions...
... done in 5.54 seconds.

Fitting dispersion trend curve...
... done in 0.60 seconds.

Fitting MAP dispersions...
... done in 5.66 seconds.

Fitting LFCs...
... done in 4.77 seconds.

Calculating cook's distance...
... done in 0.05 seconds.

Replacing 4102 outlier genes.

Fitting dispersions...
... done in 0.57 seconds.

Fitting MAP dispersions...
... done in 0.53 seconds.

Fitting LFCs...


Step 3 done


... done in 0.61 seconds.



In [9]:
# 4. Statistical Analysis
print("Extracting statistical results...")
stat_res = DeseqStats(dds, contrast=["condition", "JMML", "Control"])
stat_res.summary()

# 2. Convert results to a standard pandas DataFrame
res_df = stat_res.results_df.copy()

# 3. Create a clean mapping dictionary from your human annotation table
# (Make sure your annotation DataFrame is named 'annotation_df' or change this name)
gene_map = dict(zip(annotation_df['GeneID'].astype(str), annotation_df['Symbol']))

# 4. Map the index (NCBI Gene IDs) to Gene Symbols
res_df['Symbol'] = res_df.index.map(gene_map)

# 5. Filter for highly significant genes (adjusted p-value < 0.05)
significant_genes = res_df[res_df['padj'] < 0.05].sort_values(by="padj")

print(f"\n✅ Analysis complete! Found {len(significant_genes)} significantly changed genes (padj < 0.05).")

# 6. Show the top 15 most significant genes
print("\n--- TOP 15 DIFFERENTIALLY EXPRESSED GENES (JMML VS CONTROL) ---")
print(significant_genes[['Symbol', 'log2FoldChange', 'pvalue', 'padj']].head(15))

Extracting statistical results...


Running Wald tests...
... done in 2.02 seconds.



Log2 fold change & Wald test p-value: condition JMML vs Control
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
100287102    25.134307       -0.614886  0.609020 -1.009631  0.312672       NaN
653635      462.143360       -0.452441  0.400275 -1.130325  0.258339  0.863283
102466751    14.587879       -1.334713  0.768981 -1.735690  0.082619       NaN
107985730     1.831876        3.815736  2.845145  1.341139  0.179875       NaN
100302278     0.557227        2.113149  4.266470  0.495292  0.620394       NaN
...                ...             ...       ...       ...       ...       ...
4541        168.229691       -0.650755  0.554785 -1.172985  0.240802  0.849906
4556         65.112328       -1.037052  0.571715 -1.813930  0.069688  0.672924
4519       4045.899459       -1.242235  0.482791 -2.573026  0.010081  0.336740
4576        122.398057       -0.707437  0.559527 -1.264349  0.206105  0.826975
4571        259.963140       -0.830898  0.574496 -1.446307  0.14809

In [7]:
# Genes of Interest
GoI = ['KRAS', 'G12D', 'G13D']
GoI_df = annot.loc[annot["Symbol"].isin(GoI)]
GoI_df = GoI_df.drop(columns=['GOProcessID', 'GOComponentID', 'GOFunctionID', 'EnsemblGeneID'])

GoI_df

,GeneID,Symbol,Description,Synonyms,GeneType,Status,ChrAcc,ChrStart,ChrStop,Orientation,Length,GOFunction,GOProcess,GOComponent
23697,3845,KRAS,"KRAS proto-oncogene, GTPase",'C-K-RAS|C-K-RAS|CFC2|K-RAS2A|K-RAS2B|K-RAS4A|...,protein-coding,active,NC_000012.12,25205246,25250929,negative,5430,GTPase activity///G protein activity///protein...,MAPK cascade///positive regulation of protein ...,Golgi membrane///cytoplasm///mitochondrial out...


In [8]:
# Sanity check
# Making sure counts GeneIDs (columns) match metadata GeneIDs (columns)

assert all(counts.columns == metadata.index), "Mismatch between counts columns and metadata rows!"
print("Alignment check passed! Proceeding to differential expression analysis.")

Alignment check passed! Proceeding to differential expression analysis.
